# Azure Computer Vision - Detección de Objetos

Test simple para detectar objetos en imágenes usando **Azure Cognitive Services Computer Vision API**.

## Configuración

| Parámetro | Valor |
|-----------|-------|
| **Endpoint** | Cargado desde `.env` |
| **Región** | East US 2 |

## 1. Imports y Configuración

In [ ]:
import os
import requests
from io import BytesIO
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv()

# Credenciales desde variables de entorno
KEY = os.getenv("VISION_APIKEY")
ENDPOINT = os.getenv("VISION_ENDPOINT")

print("✅ Configuración cargada")
print(f"  📍 Endpoint: {ENDPOINT}")
print(f"  🔑 Key: {KEY[:10]}...{KEY[-5:] if KEY else 'NOT SET'}")

## 2. Test: Detectar Objetos en Imagen

In [ ]:
# Imagen de prueba
image_url = "https://learn.microsoft.com/azure/ai-services/computer-vision/media/quickstarts/presentation.png"

# API endpoint para análisis de imagen
analyze_url = f"{ENDPOINT}computervision/imageanalysis:analyze"

# Headers
headers = {
    "Ocp-Apim-Subscription-Key": KEY,
    "Content-Type": "application/json"
}

# Parámetros: detectar objetos y tags
params = {
    "api-version": "2024-02-01",
    "features": "objects,tags"
}

# Body con la URL de la imagen
data = {"url": image_url}

print(f"📷 Analizando: {image_url}")

# Llamada a la API
response = requests.post(analyze_url, headers=headers, params=params, json=data)
result = response.json()

print(f"\n✅ Status: {response.status_code}")

if response.status_code != 200:
    print(f"❌ Error: {result}")

## 3. Mostrar Resultados

In [ ]:
print("=" * 50)
print("📊 RESULTADOS")
print("=" * 50)

# Objetos
if "objectsResult" in result:
    objects = result["objectsResult"]["values"]
    print(f"\n🎯 Objetos detectados: {len(objects)}")
    for i, obj in enumerate(objects, 1):
        tags = obj["tags"][0]
        bbox = obj["boundingBox"]
        print(f"   {i}. {tags['name']} ({tags['confidence']:.1%}) - pos: ({bbox['x']}, {bbox['y']})")

# Tags
if "tagsResult" in result:
    tags_list = result["tagsResult"]["values"][:5]
    tags_str = ", ".join([t["name"] for t in tags_list])
    print(f"\n🏷️ Tags: {tags_str}")

print("\n" + "=" * 50)

## 4. Visualización con Bounding Boxes

In [ ]:
# Descargar imagen
img_response = requests.get(image_url)
img = Image.open(BytesIO(img_response.content))
draw = ImageDraw.Draw(img)

# Colores para los objetos
colors = ['#00FF00', '#FF6600', '#0066FF', '#FF00FF', '#FFFF00']

# Dibujar bounding boxes
if "objectsResult" in result:
    for i, obj in enumerate(result["objectsResult"]["values"]):
        color = colors[i % len(colors)]
        bbox = obj["boundingBox"]
        
        # Rectángulo
        x, y, w, h = bbox["x"], bbox["y"], bbox["w"], bbox["h"]
        draw.rectangle([x, y, x + w, y + h], outline=color, width=3)
        
        # Etiqueta
        tag = obj["tags"][0]
        label = f"{tag['name']} ({tag['confidence']:.0%})"
        draw.text((x, y - 15), label, fill=color)

# Mostrar
plt.figure(figsize=(12, 8))
plt.imshow(img)
num_objects = len(result.get("objectsResult", {}).get("values", []))
plt.title(f"Azure Computer Vision - {num_objects} objetos detectados")
plt.axis('off')
plt.show()

## 5. JSON Completo (Debug)

In [ ]:
import json
print(json.dumps(result, indent=2))